# Phase 2: Profile Dispatch Overhead on GPU

**Runtime:** A100 GPU (Colab Pro/Pro+)

Measures:
1. Per-layer dispatch time for each DSL policy (mean, p50, p99, max)
2. Latency distribution histogram
3. Comparison vs. actual MoE layer forward pass time on GPU
4. Overhead as % of real inference

**Prerequisites:** Run `01_trace_recording.ipynb` first to generate traces.

In [ ]:
!pip install -q torch transformers accelerate lark>=1.1 huggingface_hub

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

import sys, os
sys.path.insert(0, '/content')
DRIVE_ROOT = '/content/drive/MyDrive/moe-policylang-paper'

# Unzip moe_policylang
!unzip -qo /content/drive/MyDrive/moe-policylang-paper/moe_policylang.zip -d /content/

from moe_policylang import MoEPolicyLang
print('MoE-PolicyLang imported successfully')

In [ ]:
# Load recorded traces
import json

trace_path = f'{DRIVE_ROOT}/traces/mixtral_sample.jsonl'
with open(trace_path) as f:
    header = json.loads(f.readline())
    trace_data = [json.loads(line) for line in f]

print(f'Loaded {len(trace_data)} trace entries from {header["model_name"]}')
print(f'{header["num_layers"]} layers × {header["num_experts"]} experts (top-{header["top_k"]})')

In [ ]:
## 1. Profile Policy Dispatch Overhead

import time
import numpy as np
from moe_policylang.benchmark.policies import get_dsl_policies
from moe_policylang.compiler import compile_policy
from moe_policylang.runtime.hooks import build_hook

N = min(len(trace_data), 10000)
policies = get_dsl_policies()
results = {}

print(f'Profiling {len(policies)} policies × {N} dispatch calls each\n')
print(f'{"Policy":<25} {"Mean µs":>10} {"P50 µs":>10} {"P99 µs":>10} {"Max µs":>10}')
print('-' * 67)

for name, compiled in policies.items():
    # Warm up (100 calls)
    hook = build_hook(compiled)
    for entry in trace_data[:100]:
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))

    # Timed run — fresh hook, per-call latencies
    hook = build_hook(compiled)
    latencies = []
    for entry in trace_data[:N]:
        t0 = time.perf_counter()
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1e6)  # µs

    lat = np.array(latencies)
    results[name] = {
        'mean_us': float(np.mean(lat)),
        'p50_us': float(np.median(lat)),
        'p99_us': float(np.percentile(lat, 99)),
        'max_us': float(np.max(lat)),
        'latencies': lat,  # keep for histogram
    }
    r = results[name]
    print(f'{name:<25} {r["mean_us"]:>9.1f} {r["p50_us"]:>9.1f} {r["p99_us"]:>9.1f} {r["max_us"]:>9.1f}')

In [ ]:
## 2. Latency Distribution Histogram

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 3.5), sharey=True)
if len(results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    lat = r['latencies']
    # Clip to p99 for cleaner visualization
    clip = np.percentile(lat, 99)
    ax.hist(lat[lat <= clip], bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
    ax.axvline(r['mean_us'], color='red', linestyle='--', linewidth=1, label=f'mean={r["mean_us"]:.1f}µs')
    ax.axvline(r['p99_us'], color='orange', linestyle='--', linewidth=1, label=f'p99={r["p99_us"]:.1f}µs')
    ax.set_xlabel('Dispatch Time (µs)')
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=7)

axes[0].set_ylabel('Count')
fig.suptitle('Per-Layer Dispatch Latency Distribution', fontsize=12)
fig.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/dispatch_histogram.pdf', dpi=150)
plt.show()
print(f'Saved histogram to {DRIVE_ROOT}/results/dispatch_histogram.pdf')

## 2b. Cython Fast-Path Build & Comparison

Build the Cython-accelerated cache and scheduler extensions, then re-profile
to measure the speedup over the pure-Python dispatch path.

In [ ]:
## Build Cython extensions (including full dispatch loop)
%pip install -q cython

import subprocess, sys, importlib

# Write a temp build script — builds cache, scheduler, AND full hook loop
build_script = '/content/build_cython.py'
with open(build_script, 'w') as f:
    f.write("""
from setuptools import setup
from Cython.Build import cythonize
setup(
    name='moe_policylang_fast',
    ext_modules=cythonize([
        'moe_policylang/runtime/_fast/_cache.pyx',
        'moe_policylang/runtime/_fast/_scheduler.pyx',
        'moe_policylang/runtime/_fast/_hooks.pyx',
    ], compiler_directives={
        'boundscheck': False, 'wraparound': False,
        'cdivision': True, 'language_level': '3',
    }),
    packages=[],
    script_args=['build_ext', '--inplace'],
)
""")

result = subprocess.run([sys.executable, build_script], cwd='/content', capture_output=True, text=True)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('Cython build failed')

# Reload to pick up the newly built extensions
import moe_policylang.runtime._fast
importlib.reload(moe_policylang.runtime._fast)
from moe_policylang.runtime._fast import FAST_PATH_AVAILABLE, FAST_HOOK_AVAILABLE
print(f'\nFAST_PATH_AVAILABLE = {FAST_PATH_AVAILABLE}')
print(f'FAST_HOOK_AVAILABLE = {FAST_HOOK_AVAILABLE}')
assert FAST_PATH_AVAILABLE, 'Cython cache/scheduler modules not detected after build'
assert FAST_HOOK_AVAILABLE, 'Cython FastPolicyHook not detected after build'

In [ ]:
## 2c. Re-profile with Cython fast path and compare

# Re-compile policies (compiler will now use fast-path types)
import importlib
import moe_policylang.compiler
importlib.reload(moe_policylang.compiler)
from moe_policylang.compiler import compile_policy
from moe_policylang.runtime.hooks import build_hook
from moe_policylang.benchmark.policies import get_dsl_policies

# Rebuild policies so they use Cython caches/schedulers
fast_policies = {}
for name, compiled in get_dsl_policies().items():
    # Re-compile from IR to pick up fast path
    fast_policies[name] = compile_policy(compiled._policy_ir)

fast_results = {}
print(f'Profiling {len(fast_policies)} policies with Cython fast path × {N} calls\n')
print(f'{"Policy":<25} {"Py µs":>10} {"Cy µs":>10} {"Speedup":>10}')
print('-' * 57)

for name, compiled in fast_policies.items():
    hook = build_hook(compiled)
    # Warm up
    for entry in trace_data[:100]:
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))

    # Timed run
    hook = build_hook(compiled)
    latencies = []
    for entry in trace_data[:N]:
        t0 = time.perf_counter()
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1e6)

    lat = np.array(latencies)
    fast_results[name] = {
        'mean_us': float(np.mean(lat)),
        'p50_us': float(np.median(lat)),
        'p99_us': float(np.percentile(lat, 99)),
        'max_us': float(np.max(lat)),
    }

    py_mean = results[name]['mean_us']
    cy_mean = fast_results[name]['mean_us']
    speedup = py_mean / cy_mean if cy_mean > 0 else float('inf')
    print(f'{name:<25} {py_mean:>9.1f} {cy_mean:>9.1f} {speedup:>9.1f}x')

In [ ]:
## 2d. Python vs Cython comparison bar chart

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
width = 0.35

py_means = [results[n]['mean_us'] for n in results]
cy_means = [fast_results[n]['mean_us'] for n in results]

bars1 = ax.bar(x - width/2, py_means, width, label='Pure Python', color='#C44E52', alpha=0.85)
bars2 = ax.bar(x + width/2, cy_means, width, label='Cython Fast Path', color='#4C72B0', alpha=0.85)

ax.set_ylabel('Mean Dispatch Time (µs)')
ax.set_title('Dispatch Latency: Python vs Cython Fast Path')
ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()), rotation=15, ha='right', fontsize=9)
ax.legend()

for bar, val in zip(bars1, py_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, cy_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/python_vs_cython.pdf', dpi=150)
plt.show()
print(f'Saved to {DRIVE_ROOT}/results/python_vs_cython.pdf')

In [ ]:
## 2e. Full-Loop FastPolicyHook Benchmark
#
# Phase 3 Cython: the entire on_layer dispatch loop runs in C,
# not just the cache/scheduler components.

from moe_policylang.runtime._fast._hooks import FastPolicyHook
from moe_policylang.runtime.hooks import PolicyHook

full_hook_results = {}
print(f'Benchmarking FastPolicyHook (full Cython loop) × {N} calls\n')
print(f'{"Policy":<25} {"Python µs":>10} {"Comp. Cy µs":>12} {"Full Cy µs":>12} {"Full Speedup":>12}')
print('-' * 75)

for name, compiled in fast_policies.items():
    # Full Cython hook
    hook = FastPolicyHook(compiled)
    # Warm up
    for entry in trace_data[:100]:
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))

    # Timed run
    hook = FastPolicyHook(compiled)
    latencies = []
    for entry in trace_data[:N]:
        t0 = time.perf_counter()
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1e6)

    lat = np.array(latencies)
    full_hook_results[name] = {
        'mean_us': float(np.mean(lat)),
        'p50_us': float(np.median(lat)),
        'p99_us': float(np.percentile(lat, 99)),
        'max_us': float(np.max(lat)),
    }

    py_mean = results[name]['mean_us']
    comp_cy_mean = fast_results[name]['mean_us']
    full_cy_mean = full_hook_results[name]['mean_us']
    speedup = py_mean / full_cy_mean if full_cy_mean > 0 else float('inf')
    print(f'{name:<25} {py_mean:>9.1f} {comp_cy_mean:>11.1f} {full_cy_mean:>11.1f} {speedup:>11.2f}x')

In [ ]:
## 2f. Three-Way Comparison Chart: Python vs Component Cython vs Full Cython

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(results))
width = 0.25

py_means = [results[n]['mean_us'] for n in results]
comp_means = [fast_results[n]['mean_us'] for n in results]
full_means = [full_hook_results[n]['mean_us'] for n in results]

bars1 = ax.bar(x - width, py_means, width, label='Pure Python', color='#C44E52', alpha=0.85)
bars2 = ax.bar(x, comp_means, width, label='Component Cython', color='#55A868', alpha=0.85)
bars3 = ax.bar(x + width, full_means, width, label='Full Cython Hook', color='#4C72B0', alpha=0.85)

ax.set_ylabel('Mean Dispatch Time (µs)')
ax.set_title('Dispatch Latency: Python vs Component Cython vs Full Cython Hook')
ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()), rotation=15, ha='right', fontsize=9)
ax.legend(fontsize=8)

for bars, vals in [(bars1, py_means), (bars2, comp_means), (bars3, full_means)]:
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=7)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/python_vs_cython_3way.pdf', dpi=150)
plt.show()
print(f'Saved to {DRIVE_ROOT}/results/python_vs_cython_3way.pdf')

In [ ]:
## 3. Measure Actual MoE Layer Forward Pass Time on GPU
#
# This gives us the denominator for "overhead as % of inference time"

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'mistralai/Mixtral-8x7B-Instruct-v0.1'
print(f'Loading model for layer profiling: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map='auto',
)
print('Model loaded')

In [ ]:
# Profile individual MoE layer forward pass time
layers = model.model.layers
moe_block = layers[0].mlp

# Create a sample input matching the hidden size
hidden_size = model.config.hidden_size
sample_input = torch.randn(1, 1, hidden_size, dtype=torch.float16, device='cuda')

# Warm up
for _ in range(10):
    with torch.no_grad():
        _ = moe_block(sample_input)
torch.cuda.synchronize()

# Time individual MoE forward passes
N_FORWARD = 1000
forward_latencies = []

for _ in range(N_FORWARD):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = moe_block(sample_input)
    torch.cuda.synchronize()
    forward_latencies.append((time.perf_counter() - t0) * 1e6)  # µs

fwd = np.array(forward_latencies)
print(f'MoE Layer Forward Pass (1 token):')
print(f'  Mean:  {np.mean(fwd):.1f} µs')
print(f'  P50:   {np.median(fwd):.1f} µs')
print(f'  P99:   {np.percentile(fwd, 99):.1f} µs')

# Also time with a batch of tokens
for seq_len in [8, 32, 128]:
    batch_input = torch.randn(1, seq_len, hidden_size, dtype=torch.float16, device='cuda')
    torch.cuda.synchronize()
    times = []
    for _ in range(100):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = moe_block(batch_input)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1e6)
    print(f'  seq_len={seq_len}: mean={np.mean(times):.1f} µs, p99={np.percentile(times, 99):.1f} µs')

In [ ]:
## 4. Overhead as % of Inference

moe_fwd_mean = np.mean(fwd)  # from single-token measurement

print(f'Dispatch Overhead as % of MoE Layer Forward ({moe_fwd_mean:.0f} µs baseline)\n')
print(f'{"Policy":<25} {"Py µs":>10} {"Cy µs":>10} {"Py %":>10} {"Cy %":>10}')
print('-' * 67)

overhead_results = {}
for name, r in results.items():
    py_pct = (r['mean_us'] / moe_fwd_mean) * 100
    cy_mean = fast_results[name]['mean_us']
    cy_pct = (cy_mean / moe_fwd_mean) * 100
    print(f'{name:<25} {r["mean_us"]:>9.1f} {cy_mean:>9.1f} {py_pct:>9.2f}% {cy_pct:>9.2f}%')
    overhead_results[name] = {
        'dispatch_python_us': r['mean_us'],
        'dispatch_cython_us': cy_mean,
        'moe_forward_mean_us': float(moe_fwd_mean),
        'overhead_python_pct': round(py_pct, 2),
        'overhead_cython_pct': round(cy_pct, 2),
    }

In [ ]:
## 5. Save All Results

# Prepare serializable results (drop numpy arrays)
save_results = {}
for name, r in results.items():
    save_results[name] = {
        'mean_us': r['mean_us'],
        'p50_us': r['p50_us'],
        'p99_us': r['p99_us'],
        'max_us': r['max_us'],
    }

output = {
    'model': MODEL_NAME,
    'gpu': torch.cuda.get_device_name(0),
    'num_dispatch_calls': N,
    'moe_forward_pass': {
        'mean_us': float(np.mean(fwd)),
        'p50_us': float(np.median(fwd)),
        'p99_us': float(np.percentile(fwd, 99)),
    },
    'dispatch_latencies': {
        'python': save_results,
        'cython_components': fast_results,
        'cython_full_hook': full_hook_results,
    },
    'overhead': overhead_results,
    'speedup_components': {
        name: round(results[name]['mean_us'] / fast_results[name]['mean_us'], 2)
        for name in results if name in fast_results
    },
    'speedup_full_hook': {
        name: round(results[name]['mean_us'] / full_hook_results[name]['mean_us'], 2)
        for name in results if name in full_hook_results
    },
}

out_path = f'{DRIVE_ROOT}/results/gpu_dispatch_profile.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results saved to {out_path}')

with open('/content/gpu_dispatch_profile.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Also saved to /content/gpu_dispatch_profile.json')

In [ ]:
## Summary

After running this notebook, download `gpu_dispatch_profile.json` from Google Drive
(`moe-policylang-paper/results/`) and place it in `conference-paper/results/` for paper figure generation.

Key metrics to report in the paper:
- Per-policy dispatch latency (mean, p99) — both Python and Cython
- Cython speedup factor per policy
- Actual MoE forward pass time for context
- Overhead as % of real inference time

# (empty — delete this cell manually)